## Question 2 

In [1]:
import json
import re
from pathlib import Path
from importlib.metadata import version

import nltk
import pandas as pd
import joblib
from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)

nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)
nltk.download("punkt_tab", quiet=True)

with open("research_abstracts.json", "r", encoding="utf-8") as file:
    data = json.load(file)

df = pd.DataFrame(data)

print("Original dataset shape:", df.shape)
display(df.head())

Original dataset shape: (1138, 3)


,DOI,Abstract,Label
0,10.1155/2021/1071145,Forward-looking forecasting of the inflation r...,ai
1,10.1007/s43546-022-00384-2,Our study examined the disaggregation of infla...,ai
2,10.1002/for.2667,This paper develops a dynamic factor model tha...,classical
3,10.1007/s42973-021-00091-x,"In this paper, I use a simple SIR Macro model ...",classical
4,10.1007/s40258-012-0003-z,BackgroundGame theory is useful for identifyin...,classical


In [2]:
# Cleaning text and spliting data
df = df.drop_duplicates().reset_index(drop=True)
df["Label"] = df["Label"].str.strip().str.lower()

stop_words = set(stopwords.words("english")) - {"no", "not", "nor"}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = nltk.word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

df["Clean_Abstract"] = df["Abstract"].apply(clean_text)
df["Label_Encoded"] = df["Label"].map({"classical": 0, "ai": 1})

if df["Label_Encoded"].isna().any():
    raise ValueError("Some labels were not recognised. Check the Label column.")

X = df["Clean_Abstract"]
y = df["Label_Encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Dataset shape after removing duplicates:", df.shape)
print("\nClass distribution:")
print(df["Label"].value_counts())
print("\nTraining records:", len(X_train))
print("Testing records:", len(X_test))
print("\nMissing cleaned abstracts:", X.isna().sum())
print("\nSample cleaned abstract:")
print(X.iloc[0])

Dataset shape after removing duplicates: (1136, 5)

Class distribution:
Label
classical    900
ai           236
Name: count, dtype: int64

Training records: 908
Testing records: 228

Missing cleaned abstracts: 0

Sample cleaned abstract:
forward looking forecasting inflation rate could help central bank government departments better use monetary policy stabilize prices prevent impact inflation market entities especially low middle income groups also help financial institutions investors better make investment decisions sense forecast inflation rate great significance existing literature mainly uses linear models autoregressive ar vector autoregressive var models predict inflation rate nonlinear relationship variables mining historical data information relatively lacking therefore prediction strategies accuracy existing literature need improved predictive model designed deep learning fully mine nonlinear relationship variables process complex long term time series dynamic information th

In [3]:
#2.1 Training four models
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Support Vector Machine": LinearSVC(random_state=42),
    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=42,
        n_jobs=-1,
    ),
}

results = []
trained_models = {}

for name, classifier in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=20000,
            ngram_range=(1, 2),
            sublinear_tf=True,
        )),
        ("classifier", classifier),
    ])

    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    trained_models[name] = pipeline

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "AI Precision": precision_score(y_test, predictions, pos_label=1, zero_division=0),
        "AI Recall": recall_score(y_test, predictions, pos_label=1, zero_division=0),
        "AI F1-Score": f1_score(y_test, predictions, pos_label=1, zero_division=0),
    })

    print(f"\n{'=' * 60}\n{name}\n{'=' * 60}")
    print(classification_report(
        y_test,
        predictions,
        labels=[0, 1],
        target_names=["Classical", "AI"],
        zero_division=0,
    ))
    print("Confusion matrix:")
    print(confusion_matrix(y_test, predictions, labels=[0, 1]))

results_df = pd.DataFrame(results).sort_values(
    by="AI F1-Score",
    ascending=False,
).reset_index(drop=True)

print("\nModel performance comparison:")
display(results_df)


Multinomial Naive Bayes
              precision    recall  f1-score   support

   Classical       0.80      1.00      0.89       181
          AI       1.00      0.02      0.04        47

    accuracy                           0.80       228
   macro avg       0.90      0.51      0.46       228
weighted avg       0.84      0.80      0.71       228

Confusion matrix:
[[181   0]
 [ 46   1]]

Logistic Regression
              precision    recall  f1-score   support

   Classical       0.84      1.00      0.91       181
          AI       1.00      0.26      0.41        47

    accuracy                           0.85       228
   macro avg       0.92      0.63      0.66       228
weighted avg       0.87      0.85      0.81       228

Confusion matrix:
[[181   0]
 [ 35  12]]


C:\Users\Tbose\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(



Support Vector Machine
              precision    recall  f1-score   support

   Classical       0.93      0.98      0.95       181
          AI       0.92      0.70      0.80        47

    accuracy                           0.93       228
   macro avg       0.92      0.84      0.87       228
weighted avg       0.92      0.93      0.92       228

Confusion matrix:
[[178   3]
 [ 14  33]]

Random Forest
              precision    recall  f1-score   support

   Classical       0.89      1.00      0.94       181
          AI       1.00      0.51      0.68        47

    accuracy                           0.90       228
   macro avg       0.94      0.76      0.81       228
weighted avg       0.91      0.90      0.89       228

Confusion matrix:
[[181   0]
 [ 23  24]]

Model performance comparison:


,Model,Accuracy,AI Precision,AI Recall,AI F1-Score
0,Support Vector Machine,0.925439,0.916667,0.702128,0.795181
1,Random Forest,0.899123,1.000000,0.510638,0.676056
2,Logistic Regression,0.846491,1.000000,0.255319,0.406780
3,Multinomial Naive Bayes,0.798246,1.000000,0.021277,0.041667


In [4]:
#2.2 Evaluating balanced SVM
balanced_svm = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        sublinear_tf=True,
    )),
    ("classifier", LinearSVC(
        class_weight="balanced",
        random_state=42,
    )),
])

balanced_svm.fit(X_train, y_train)
balanced_predictions = balanced_svm.predict(X_test)

original_svm = trained_models["Support Vector Machine"]
original_predictions = original_svm.predict(X_test)

comparison_df = pd.DataFrame([
    {
        "Model": "Original SVM",
        "Accuracy": accuracy_score(y_test, original_predictions),
        "AI Precision": precision_score(y_test, original_predictions, pos_label=1, zero_division=0),
        "AI Recall": recall_score(y_test, original_predictions, pos_label=1, zero_division=0),
        "AI F1-Score": f1_score(y_test, original_predictions, pos_label=1, zero_division=0),
    },
    {
        "Model": "Enhanced SVM - Balanced Class Weights",
        "Accuracy": accuracy_score(y_test, balanced_predictions),
        "AI Precision": precision_score(y_test, balanced_predictions, pos_label=1, zero_division=0),
        "AI Recall": recall_score(y_test, balanced_predictions, pos_label=1, zero_division=0),
        "AI F1-Score": f1_score(y_test, balanced_predictions, pos_label=1, zero_division=0),
    },
])

display(comparison_df)

print("\nEnhanced SVM classification report:")
print(classification_report(
    y_test,
    balanced_predictions,
    labels=[0, 1],
    target_names=["Classical", "AI"],
    zero_division=0,
))

print("Enhanced SVM confusion matrix:")
print(confusion_matrix(y_test, balanced_predictions, labels=[0, 1]))

C:\Users\Tbose\anaconda3\Lib\site-packages\sklearn\svm\_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(


,Model,Accuracy,AI Precision,AI Recall,AI F1-Score
0,Original SVM,0.925439,0.916667,0.702128,0.795181
1,Enhanced SVM - Balanced Class Weights,0.934211,0.900000,0.765957,0.827586



Enhanced SVM classification report:
              precision    recall  f1-score   support

   Classical       0.94      0.98      0.96       181
          AI       0.90      0.77      0.83        47

    accuracy                           0.93       228
   macro avg       0.92      0.87      0.89       228
weighted avg       0.93      0.93      0.93       228

Enhanced SVM confusion matrix:
[[177   4]
 [ 11  36]]


In [5]:
#2.3 Algorithmic configuration
# Saving the model
import joblib

deployment_folder = Path("deployment")
deployment_folder.mkdir(exist_ok=True)

model_path = deployment_folder / "balanced_svm_model.pkl"
joblib.dump(balanced_svm, model_path)

print(f"Model saved to: {model_path.resolve()}")

Model saved to: C:\Users\Tbose\deployment\balanced_svm_model.pkl


In [6]:
%%writefile deployment/classifier.py
import re
from pathlib import Path

import joblib
import nltk
import streamlit as st
from nltk.corpus import stopwords

nltk.download("stopwords", quiet=True)
STOP_WORDS = set(stopwords.words("english")) - {"no", "not", "nor"}

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    tokens = text.split()
    tokens = [word for word in tokens if word not in STOP_WORDS]
    return " ".join(tokens)

st.set_page_config(page_title="Abstract Classifier", page_icon="📄")

model_path = Path(__file__).parent / "balanced_svm_model.pkl"
model = joblib.load(model_path)

st.title("Research Abstract Classifier")
st.write("Classify a research abstract as AI-generated or Classical.")

abstract = st.text_area(
    "Paste the research abstract below:",
    height=250,
    placeholder="Enter the abstract text here...",
)

if st.button("Classify"):
    if not abstract.strip():
        st.warning("Please enter an abstract first.")
    else:
        cleaned_abstract = clean_text(abstract)
        prediction = int(model.predict([cleaned_abstract])[0])

        if prediction == 1:
            st.subheader("Prediction: AI")
        else:
            st.subheader("Prediction: Classical")

Overwriting deployment/classifier.py


In [7]:
#Requirements and readme
requirements = [
    f"streamlit=={version('streamlit')}",
    f"scikit-learn=={version('scikit-learn')}",
    f"joblib=={version('joblib')}",
    f"nltk=={version('nltk')}",
]

(deployment_folder / "requirements.txt").write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8",
)

readme_lines = [
    "# Research Abstract Classifier",
    "",
    "A Streamlit app that classifies research abstracts as AI-generated or Classical using a balanced SVM model.",
    "",
    "## Files",
    "",
    "- `classifier.py` — Streamlit web app",
    "- `balanced_svm_model.pkl` — fitted TF-IDF and balanced SVM pipeline",
    "- `requirements.txt` — package versions used by the app",
    "",
    "## Run locally",
    "",
    "Open Anaconda Prompt from the folder containing `deployment`, then run:",
    "",
    "```bash",
    "cd deployment",
    "pip install -r requirements.txt",
    "streamlit run classifier.py",
    "```",
]

(deployment_folder / "README.md").write_text(
    "\n".join(readme_lines) + "\n",
    encoding="utf-8",
)

print("Created requirements.txt and README.md in the deployment folder.")

Created requirements.txt and README.md in the deployment folder.


In [ ]:
!streamlit run deployment/classifier.py --server.port 8502